# Aula 09 - Problemas Comuns com Modelagem de IA e mais Feature Engineering

**Módulo 03 IN** - Lógica para predição com inteligência artificial
**15/09/2026 - Sprint 4 - Prof. Ovidio Lopes da Cruz Netto**

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/canaldoovidio/2026-2A-M03/blob/main/notebooks/aula09.ipynb)

## O que este notebook é

As Aulas 04 a 08 construíram a base analítica mensal, o primeiro modelo do case, os agrupamentos
e a redução de dimensionalidade. Este notebook mede cinco erros que atravessam esse caminho
inteiro e não aparecem na métrica que a ferramenta devolve por padrão: o corte aleatório que
melhora o MAPE de quem memoriza vizinho, a acurácia que premia o classificador que prevê sempre a
mesma classe, a entropia que separa o corte útil do irrelevante, a imputação que erra 34,53% sem
reclamar e a dimensionalidade que afasta os vizinhos.

Todo número aqui vem dos CSVs de `dados/mensal/`, e cada conclusão está travada em
`tools/tests/test_problemas_aula09.py`. O escopo da aula, o corte de conteúdo e as duas correções
de roteiro estão em `docs/adrs/ADR-012`.

## Ao final deste notebook você terá

1. remontado a base analítica mensal da Aula 07, com 339 linhas, 315 meses de treino e 24 de
   teste;
2. medido os quatro modelos nos dois cortes e visto o MAPE melhorar em três deles e piorar no
   quarto, com a ressalva de escala do RMSE demonstrada antes;
3. contado, mês a mês, quantos meses de teste ganham vizinho no treino em cada corte;
4. construído o alvo binário do case e medido a baseline majoritária contra cinco
   classificadores, com a matriz de confusão de cada um;
5. calculado a entropia da raiz e o ganho de informação de cada corte candidato à mão, e
   conferido com a árvore de entropia do scikit-learn;
6. medido a ausência de período das cinco séries e comparado cinco estratégias de imputação
   contra o valor real escondido;
7. medido o efeito de ir de 2 para 11 features sobre a distância média entre meses, o KNN e a
   regressão linear.

## 1. A base analítica mensal da Aula 07

A célula abaixo lê as cinco séries mensais de `dados/mensal/` e remonta a base analítica da
Aula 07. A definição é a mesma das Aulas 07 e 08, reimplementada aqui porque o notebook precisa
rodar sozinho no Colab, sem depender de nenhum outro arquivo do repositório:

- junção interna das cinco séries por `periodo`, em ordem cronológica;
- `dias` do mês civil, `sen` e `cos` da posição no ciclo anual;
- `lag1`, `lag2`, `lag3` e `lag12` do abate de frangos, mais as outras quatro séries defasadas em
  um mês;
- descarte das linhas com valor ausente, o que deixa **339 linhas** de 1998-01 a 2026-03;
- corte por data: **315 meses de treino** (1998-01 a 2024-03) e **24 meses de teste** (2024-04 a
  2026-03), que é o horizonte que a LDC pediu no TAPI.

A célula traz um `try`/`except` para o caso de a rede da sala cair no meio da leitura: se a
internet falhar e o arquivo não estiver na pasta local, a mensagem de erro orienta a pedir a
pasta `dados` para uma dupla vizinha.

In [ ]:
import calendar
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

SERIES = [
    "abate_bovinos",
    "abate_suinos",
    "abate_frangos",
    "producao_ovos",
    "producao_leite",
]
ALVO = "abate_frangos"
FEATURES = (["lag1", "lag2", "lag3", "lag12", "sen", "cos", "dias"]
            + [serie + "_lag1" for serie in SERIES if serie != ALVO])
N_TESTE = 24
SEMENTE = 42
SEMENTES_SORTEIO = 10

MENSAL_LOCAL = os.path.join("..", "dados", "mensal")
MENSAL_BRUTA = ("https://raw.githubusercontent.com/canaldoovidio/2026-2A-M03/"
                "main/dados/mensal/")

caminhos = {}
for nome in SERIES:
    arquivo = nome + ".csv"
    local = os.path.join(MENSAL_LOCAL, arquivo)
    if os.path.exists(local):
        caminhos[nome] = local
    else:
        if not os.path.exists(arquivo):
            try:
                urllib.request.urlretrieve(MENSAL_BRUTA + arquivo, arquivo)
            except Exception as erro:
                raise RuntimeError(
                    "Nao foi possivel baixar '%s' pela internet (%s). "
                    "Se a rede da sala falhou, peca a pasta 'dados' para uma dupla "
                    "que tenha o repositorio clonado no computador (ela fica na raiz "
                    "do repositorio) e coloque essa pasta ao lado deste notebook. "
                    "Depois, rode esta celula de novo." % (arquivo, erro)
                ) from erro
        caminhos[nome] = arquivo

series_cruas = {nome: pd.read_csv(caminhos[nome])[["periodo", "valor"]]
                for nome in SERIES}

base = None
for nome in SERIES:
    coluna = series_cruas[nome].rename(columns={"valor": nome})
    base = coluna if base is None else base.merge(coluna, on="periodo", how="inner")
base = base.sort_values("periodo").reset_index(drop=True)

base["mes"] = base["periodo"].str[-2:].astype(int)
base["dias"] = [calendar.monthrange(int(p[:4]), int(p[-2:]))[1] for p in base["periodo"]]
base["sen"] = np.sin(2 * np.pi * base["mes"] / 12)
base["cos"] = np.cos(2 * np.pi * base["mes"] / 12)
for k in (1, 2, 3, 12):
    base["lag%d" % k] = base[ALVO].shift(k)
for nome in SERIES:
    if nome != ALVO:
        base[nome + "_lag1"] = base[nome].shift(1)
base = base.dropna().reset_index(drop=True)

CORTE = len(base) - N_TESTE
X = base[FEATURES].to_numpy(dtype=float)
y = base[ALVO].to_numpy(dtype=float)
lag12 = base["lag12"].to_numpy(dtype=float)

print("linhas: %d   de %s a %s" % (len(base), base["periodo"].iloc[0], base["periodo"].iloc[-1]))
print("treino: %d meses (%s a %s)" % (CORTE, base["periodo"].iloc[0], base["periodo"].iloc[CORTE - 1]))
print("teste:  %d meses (%s a %s)" % (N_TESTE, base["periodo"].iloc[CORTE], base["periodo"].iloc[-1]))
print("features (%d): %s" % (len(FEATURES), ", ".join(FEATURES)))

## 2. Vazamento temporal: os dois cortes, lado a lado

O corte por data separa os 24 últimos meses, na ordem em que aconteceram, e é o protocolo do
acervo desde a `ADR-008`. O sorteio aleatório embaralha essa ordem e manda meses antigos para o
teste enquanto deixa meses recentes no treino.

Antes de comparar qualquer métrica de erro, a célula abaixo mede o **alvo médio** de cada
conjunto de teste. É a ressalva que o roteiro original desta aula não fazia: o teste sorteado cai
majoritariamente em anos antigos, quando o abate era menor, e o RMSE em quilogramas cai junto,
mesmo sem o modelo ter ganhado nada.

In [ ]:
def mape(real, previsto):
    return float(np.mean(np.abs((real - previsto) / real)) * 100)


def rmse(real, previsto):
    return float(np.sqrt(np.mean((real - previsto) ** 2)))


def indices(aleatorio, semente=0):
    """Indices de treino e teste, por corte de data ou por sorteio."""
    if not aleatorio:
        todos = np.arange(len(base))
        return todos[:CORTE], todos[CORTE:]
    return train_test_split(np.arange(len(base)),
                            test_size=N_TESTE / len(base), random_state=semente)


media_data = float(y[CORTE:].mean())
medias_sorteio = []
for semente in range(SEMENTES_SORTEIO):
    _, ite = indices(True, semente)
    medias_sorteio.append(float(y[ite].mean()))
media_sorteio = float(np.mean(medias_sorteio))

print("alvo medio do teste por data:    %15.0f kg" % media_data)
print("alvo medio do teste sorteado:    %15.0f kg  (media de %d sementes)"
      % (media_sorteio, SEMENTES_SORTEIO))
print("o teste sorteado e %.1f%% menor em escala" % ((1 - media_sorteio / media_data) * 100))

Com escalas tão diferentes, comparar RMSE entre os dois cortes mistura duas coisas num número só.
A célula abaixo mostra isso de propósito, com a regressão linear em nível: o RMSE cai no sorteio
mesmo para um modelo que, medido por MAPE, não melhora.

A partir daqui a leitura usa **MAPE**, que é razão entre erro e valor real e não depende de
escala.

In [ ]:
def avaliar(fabrica, aleatorio, em_razao=False):
    """MAPE medio de teste, com corte por data ou media de dez sorteios."""
    alvo = y / lag12 if em_razao else y
    saida = []
    for semente in range(SEMENTES_SORTEIO if aleatorio else 1):
        itr, ite = indices(aleatorio, semente)
        escalador = StandardScaler().fit(X[itr])
        modelo = fabrica().fit(escalador.transform(X[itr]), alvo[itr])
        previsto = modelo.predict(escalador.transform(X[ite]))
        if em_razao:
            previsto = previsto * lag12[ite]
        saida.append(mape(y[ite], previsto))
    return float(np.mean(saida))


def avaliar_rmse(fabrica, aleatorio):
    saida = []
    for semente in range(SEMENTES_SORTEIO if aleatorio else 1):
        itr, ite = indices(aleatorio, semente)
        escalador = StandardScaler().fit(X[itr])
        modelo = fabrica().fit(escalador.transform(X[itr]), y[itr])
        saida.append(rmse(y[ite], modelo.predict(escalador.transform(X[ite]))))
    return float(np.mean(saida))


print("RMSE, regressao linear em nivel")
print("  corte por data: %15.0f" % avaliar_rmse(lambda: LinearRegression(), False))
print("  sorteio:        %15.0f" % avaliar_rmse(lambda: LinearRegression(), True))
print()
print("RMSE, KNN k=5 em nivel")
print("  corte por data: %15.0f" % avaliar_rmse(lambda: KNeighborsRegressor(n_neighbors=5), False))
print("  sorteio:        %15.0f" % avaliar_rmse(lambda: KNeighborsRegressor(n_neighbors=5), True))

In [ ]:
modelos = [
    ("KNN k=5", lambda: KNeighborsRegressor(n_neighbors=5), False),
    ("arvore d=3", lambda: DecisionTreeRegressor(max_depth=3, random_state=SEMENTE), False),
    ("random forest 300", lambda: RandomForestRegressor(n_estimators=300, random_state=SEMENTE), False),
    ("regressao, razao", lambda: LinearRegression(), True),
]

print("%-20s %14s %14s %12s" % ("modelo", "MAPE por data", "MAPE sorteio", "diferenca"))
for rotulo, fabrica, em_razao in modelos:
    honesto = avaliar(fabrica, False, em_razao)
    sorteado = avaliar(fabrica, True, em_razao)
    print("%-20s %13.2f%% %13.2f%% %11.2f" % (rotulo, honesto, sorteado, honesto - sorteado))

Três modelos melhoram com o sorteio, e um piora.

O KNN, a árvore e a floresta preveem por semelhança com observações vistas no treino, e ganhar
vizinhos temporais é ganhar exatamente o tipo de informação que eles usam. A regressão linear
sobre a razão estima onze coeficientes e não guarda observação nenhuma: para ela o sorteio só
troca um teste concentrado no regime recente por um teste espalhado por 28 anos, e o MAPE piora
de 3,32% para 3,60%.

O contraste é o que impede a generalização errada. "Sorteio infla a métrica" é falso como regra
geral nesta base. O que vale é disciplina de protocolo: corte por data antes de saber qual modelo
vai ganhar.

In [ ]:
def com_vizinho(itr, ite):
    """Quantos meses de teste tem o mes anterior ou o seguinte dentro do treino."""
    treino = set(itr.tolist())
    return sum(1 for t in ite.tolist() if (t - 1) in treino or (t + 1) in treino)


itr, ite = indices(False)
print("corte por data: %d de %d meses de teste tem vizinho no treino" % (com_vizinho(itr, ite), N_TESTE))

fracoes = []
for semente in range(SEMENTES_SORTEIO):
    itr, ite = indices(True, semente)
    quantos = com_vizinho(itr, ite)
    fracoes.append(quantos / len(ite))
    if semente < 3:
        print("sorteio, semente %d: %d de %d" % (semente, quantos, N_TESTE))
print("sorteio, media das %d sementes: %.1f%%" % (SEMENTES_SORTEIO, np.mean(fracoes) * 100))

É o mecanismo inteiro, contado mês a mês. No corte por data só o primeiro mês do teste tem
vizinho, porque ele encosta no fim do treino, e nos outros 23 o modelo precisa extrapolar. Com o
sorteio, quase todo mês de teste fica entre dois meses conhecidos, e prever passa a ser
interpolar. Séries de abate mudam pouco de um mês para o outro, então conhecer os dois vizinhos é
quase conhecer a resposta.

## 3. Classificação: o alvo binário do case

Matriz de confusão, precisão, revocação, Naive Bayes, regressão logística e SVM precisam de um
alvo categórico, e os três modelos do TAPI preveem quantidade. O alvo abaixo é o mais direto que
as cinco séries permitem, usa uma coluna que a base já tinha e não inventa nenhum valor: **o
abate do mês supera o mesmo mês do ano anterior**.

Ele existe para ensinar as métricas de classificação sobre dado real. Os três modelos do case
seguem sendo de regressão, e a ART.7 compara modelos de regressão.

In [ ]:
alvo_binario = (y > lag12).astype(int)
atr, ate = alvo_binario[:CORTE], alvo_binario[CORTE:]

escalador = StandardScaler().fit(X[:CORTE])
Ztr, Zte = escalador.transform(X[:CORTE]), escalador.transform(X[CORTE:])

print("positivos na base inteira: %3d de %3d  (%.1f%%)"
      % (alvo_binario.sum(), len(alvo_binario), alvo_binario.mean() * 100))
print("positivos no treino:       %3d de %3d  (%.1f%%)" % (atr.sum(), len(atr), atr.mean() * 100))
print("positivos no teste:        %3d de %3d  (%.1f%%)" % (ate.sum(), len(ate), ate.mean() * 100))

maioria = np.full(len(ate), int(atr.mean() > 0.5))
print()
print("baseline que preve sempre a classe majoritaria")
print("  acuracia:  %.4f" % accuracy_score(ate, maioria))
print("  precisao:  %.4f" % precision_score(ate, maioria))
print("  revocacao: %.4f" % recall_score(ate, maioria))
print("  F1:        %.4f" % f1_score(ate, maioria))

Revocação 1,000 é consequência aritmética de responder "cresce" em todos os meses: quem chama
tudo de alta encontra todas as altas. Um número perfeito, produzido por um modelo que não olha
para nenhuma feature. É por isso que a baseline precisa ser declarada antes de qualquer
comparação: sem ela, 83,3% de acurácia parece bom resultado; com ela, 83,3% é o piso.

In [ ]:
classificadores = {
    "logistica": LogisticRegression(max_iter=2000, random_state=SEMENTE),
    "naive bayes": GaussianNB(),
    "SVM linear": SVC(kernel="linear", random_state=SEMENTE),
    "SVM RBF": SVC(kernel="rbf", random_state=SEMENTE),
    "arvore entropia": DecisionTreeClassifier(criterion="entropy", max_depth=3,
                                              random_state=SEMENTE),
}

print("%-16s %9s %9s %10s %7s" % ("classificador", "acuracia", "precisao", "revocacao", "F1"))
print("%-16s %8.1f%% %9.3f %10.3f %7.3f"
      % ("baseline", accuracy_score(ate, maioria) * 100, precision_score(ate, maioria),
         recall_score(ate, maioria), f1_score(ate, maioria)))

previsoes = {}
for nome, modelo in classificadores.items():
    previsto = modelo.fit(Ztr, atr).predict(Zte)
    previsoes[nome] = previsto
    print("%-16s %8.1f%% %9.3f %10.3f %7.3f"
          % (nome, accuracy_score(ate, previsto) * 100,
             precision_score(ate, previsto, zero_division=0),
             recall_score(ate, previsto, zero_division=0),
             f1_score(ate, previsto, zero_division=0)))

print()
for nome in ("logistica", "SVM RBF", "naive bayes"):
    m = confusion_matrix(ate, previsoes[nome])
    print("matriz de confusao, %s   (linhas: real; colunas: previsto)" % nome)
    print("  queda real: %2d %2d" % (m[0][0], m[0][1]))
    print("  alta real:  %2d %2d" % (m[1][0], m[1][1]))

Nenhum dos cinco supera a acurácia da baseline.

SVM RBF e a árvore repetem as quatro métricas dela porque fazem a mesma coisa: preveem alta nos
24 meses. A regressão logística é a única que decide algo, acerta 2 dos 4 meses de queda que
ninguém mais encontra, e paga com 3 meses de alta classificados como queda, o que a leva a 79,2%,
abaixo da baseline. O Naive Bayes gaussiano prevê queda em quase todos os meses e termina com
precisão e revocação zero na classe positiva: ele supõe independência entre as features dada a
classe, e a Aula 08 mediu que 20 dos 55 pares dessas colunas passam de 0,9 de correlação
absoluta.

A conclusão é sobre a métrica: acurácia sozinha não distingue um modelo que decide de um que
responde sempre a mesma coisa, e é a matriz de confusão que separa os dois.

## 4. Entropia passo a passo

Entropia mede a incerteza de um nó em bits. O ganho de informação de um corte é a entropia do nó
menos a média das entropias dos dois filhos, ponderada pelo número de observações de cada lado, e
é o que a árvore com `criterion="entropy"` maximiza em cada nó.

A célula abaixo faz a conta à mão, varrendo todos os limiares de todas as onze features nos 315
meses de treino, sem padronizar.

In [ ]:
def entropia(p):
    if p <= 0 or p >= 1:
        return 0.0
    return float(-p * np.log2(p) - (1 - p) * np.log2(1 - p))


raiz = entropia(float(atr.mean()))
print("raiz: %d positivos em %d, p = %.4f, entropia = %.4f bits"
      % (atr.sum(), len(atr), atr.mean(), raiz))


def melhor_ganho(coluna):
    """Maior ganho de informacao entre os limiares de uma coluna."""
    valores = np.unique(coluna)
    limiares = (valores[:-1] + valores[1:]) / 2
    melhor = (None, None)
    for t in limiares:
        esq, dir_ = atr[coluna <= t], atr[coluna > t]
        if len(esq) == 0 or len(dir_) == 0:
            continue
        ponderada = (len(esq) * entropia(float(esq.mean()))
                     + len(dir_) * entropia(float(dir_.mean()))) / len(atr)
        if melhor[0] is None or raiz - ponderada > melhor[0]:
            melhor = (raiz - ponderada, float(t))
    return melhor


ganhos = {f: melhor_ganho(base[f].to_numpy(dtype=float)[:CORTE]) for f in FEATURES}
print()
print("%-22s %10s %20s" % ("feature", "ganho", "limiar"))
for f in sorted(ganhos, key=lambda f: -ganhos[f][0]):
    print("%-22s %9.4f %20.0f" % (f, ganhos[f][0], ganhos[f][1]))

In [ ]:
dias_treino = base["dias"].to_numpy(dtype=float)[:CORTE]
print("valores distintos de dias no treino:", sorted(set(dias_treino.tolist())))
for limiar in (31.0, 30.5, 28.5):
    esq = int((dias_treino <= limiar).sum())
    print("dias <= %4.1f  ->  %3d contra %3d meses" % (limiar, esq, len(dias_treino) - esq))

arvore = DecisionTreeClassifier(criterion="entropy", max_depth=3,
                                random_state=SEMENTE).fit(X[:CORTE], atr)
no = arvore.tree_
esquerdo, direito = no.children_left[0], no.children_right[0]
print()
print("a arvore escolhe na raiz: %s <= %.0f" % (FEATURES[no.feature[0]], no.threshold[0]))
print("  entropia da raiz:            %.4f bits com %d meses" % (no.impurity[0], no.n_node_samples[0]))
print("  filho da esquerda:           %.4f bits com %d meses"
      % (no.impurity[esquerdo], no.n_node_samples[esquerdo]))
print("  filho da direita:            %.4f bits com %d meses"
      % (no.impurity[direito], no.n_node_samples[direito]))
print("  conta a mao, mesmo corte:    %s <= %.0f, ganho de %.4f bits"
      % ("lag12", ganhos["lag12"][1], ganhos["lag12"][0]))

A conta à mão e o `DecisionTreeClassifier` chegam ao mesmo corte, com dois quilogramas de
diferença no limiar, que vem do arredondamento do ponto médio entre os mesmos dois valores
consecutivos.

O corte `dias <= 31` deixa os 315 meses do mesmo lado: um nó que não parte a amostra tem a
entropia do pai como entropia do filho, e ganho zero por construção. É por isso que a varredura
só considera pontos médios entre valores distintos e consecutivos.

O filho da esquerda reúne os meses em que o abate de doze meses antes era baixo, e é o lado mais
previsível, com 0,4374 bits contra 0,9123 do outro.

## 5. Ausência de dado: o que a junção interna descarta

O roteiro original desta aula pedia `SimpleImputer` "nos vazios da própria série". Nenhum dos dez
CSVs versionados tem valor vazio: a ausência real do acervo é de período, porque `producao_ovos`
começa dez anos antes das outras quatro séries.

In [ ]:
periodos = {nome: set(df["periodo"]) for nome, df in series_cruas.items()}
uniao = set.union(*periodos.values())
intersecao = set.intersection(*periodos.values())

print("uniao dos periodos:    %3d meses (%s a %s)" % (len(uniao), min(uniao), max(uniao)))
print("intersecao:            %3d meses (%s a %s)" % (len(intersecao), min(intersecao), max(intersecao)))
print("descartados pela juncao interna: %d meses (%.1f%%)"
      % (len(uniao) - len(intersecao), (len(uniao) - len(intersecao)) / len(uniao) * 100))

celulas = len(uniao) * len(SERIES)
vazias = celulas - sum(len(p) for p in periodos.values())
print("celulas da matriz periodo por serie: %d, das quais %d vazias (%.1f%%)"
      % (celulas, vazias, vazias / celulas * 100))
print()
for nome in SERIES:
    print("%-16s %3d meses medidos, de %s" % (nome, len(periodos[nome]), min(periodos[nome])))
print()
print("valores vazios nos cinco CSVs:",
      sum(int(df["valor"].isna().sum()) for df in series_cruas.values()))

Como a base não tem vazio para imputar, o exercício é invertido: esconder valores conhecidos e
medir cada estratégia contra a verdade. A célula abaixo mascara 16 meses do treino (5% de 315),
sorteados com semente 42, e compara cinco estratégias.

In [ ]:
alvo_treino = y[:CORTE].copy()
gerador = np.random.default_rng(SEMENTE)
quantos = int(round(0.05 * CORTE))
mascarados = np.sort(gerador.choice(np.arange(1, CORTE - 1), size=quantos, replace=False))
verdade = alvo_treino[mascarados]

com_furo = alvo_treino.copy()
com_furo[mascarados] = np.nan

ultimo = com_furo.copy()
for i in range(1, len(ultimo)):
    if np.isnan(ultimo[i]):
        ultimo[i] = ultimo[i - 1]

validos = ~np.isnan(com_furo)
fator = float(np.nanmean(com_furo[12:] / alvo_treino[:CORTE - 12]))
sazonal = np.array([alvo_treino[i - 12] * fator for i in mascarados])

estrategias = {
    "media da serie": np.full(quantos, np.nanmean(com_furo)),
    "mediana da serie": np.full(quantos, np.nanmedian(com_furo)),
    "ultimo valor medido": ultimo[mascarados],
    "interpolacao linear": np.interp(mascarados, np.arange(CORTE)[validos], com_furo[validos]),
    "mesmo mes do ano anterior": sazonal,
}

print("%d meses mascarados de %d, semente %d, fator medio %.5f"
      % (quantos, CORTE, SEMENTE, fator))
print()
for nome, imputado in sorted(estrategias.items(),
                             key=lambda kv: -np.mean(np.abs((kv[1] - verdade) / verdade))):
    erro = 100 * float(np.mean(np.abs((imputado - verdade) / verdade)))
    print("%-28s erro medio de %6.2f%%" % (nome, erro))

Média e mediana erram cerca de 34% porque a série cresce por 28 anos: a média global fica entre o
abate de 1998 e o de 2026 e não descreve nenhum mês em particular. As duas estratégias que olham
para vizinhos erram cerca de 7%. A que erra menos usa a estrutura sazonal já medida nas Aulas 04
a 08, e ela não vem pronta em nenhum imputador: precisa ser escrita.

O que se afirma aqui é a **ordem**, e a razão dela, não o valor exato de cada estratégia: os
cinco números dependem de quais 16 meses o sorteio escolheu. Mascarar em bloco contíguo muda a
conclusão, porque a interpolação linear perde os vizinhos medidos e passa a errar como a média.

## 6. Dimensionalidade: a distância cresce e os dois modelos divergem

A última medição vai de 2 para 11 features, com o mesmo alvo em razão e o escalador ajustado só
no treino. A distância é a média entre todos os pares de meses de treino na matriz padronizada.

In [ ]:
conjuntos = {
    2: ["lag1", "lag12"],
    4: ["lag1", "lag12", "sen", "cos"],
    7: ["lag1", "lag2", "lag3", "lag12", "sen", "cos", "dias"],
    11: FEATURES,
}
razao = y / lag12

print("%9s %14s %10s %12s" % ("features", "distancia", "KNN k=5", "regressao"))
for quantas, colunas in conjuntos.items():
    matriz = base[colunas].to_numpy(dtype=float)
    esc = StandardScaler().fit(matriz[:CORTE])
    Ztr_d, Zte_d = esc.transform(matriz[:CORTE]), esc.transform(matriz[CORTE:])

    pares = np.linalg.norm(Ztr_d[:, None, :] - Ztr_d[None, :, :], axis=2)
    distancia = float(pares[~np.eye(len(pares), dtype=bool)].mean())

    p_knn = KNeighborsRegressor(n_neighbors=5).fit(Ztr_d, razao[:CORTE]).predict(Zte_d) * lag12[CORTE:]
    p_reg = LinearRegression().fit(Ztr_d, razao[:CORTE]).predict(Zte_d) * lag12[CORTE:]

    print("%9d %14.2f %9.2f%% %11.2f%%"
          % (quantas, distancia, mape(y[CORTE:], p_knn), mape(y[CORTE:], p_reg)))

Cada coluna nova soma um termo positivo dentro da raiz da distância euclidiana, então toda
distância cresce e os vizinhos ficam todos mais longe: é a maldição de dimensionalidade do
autoestudo da semana. O KNN decide por vizinhança e piora de 3,71% para 5,01%. A regressão linear
estima um coeficiente por coluna, e cada coluna nova é uma chance de explicar o alvo: ela melhora
de 4,71% para 3,32%, que é o modelo do fecho da Aula 07.

A tabela não diz que menos features é melhor. O que cresce sempre é a distância média; os dois
modelos reagem em direções opostas ao mesmo crescimento.

## O que levar para a ART.7

**ART.7 Comparação de modelos, peso 8**, é a entrega da Sprint 4, com planning em 14/09 e review
em 25/09. O que esta aula entrega é o protocolo sob o qual a comparação precisa acontecer:

1. **corte por data em todo candidato**, sempre o mesmo, decidido antes de saber qual modelo vai
   ganhar;
2. **baseline declarada** ao lado de cada métrica, e matriz de confusão sempre que o alvo for
   categórico;
3. **métrica sem dependência de escala** quando os conjuntos comparados cobrirem períodos
   diferentes;
4. **estratégia de imputação medida** contra valor conhecido, e não escolhida por hábito;
5. **número de features declarado** junto com a família de modelo, porque o efeito de acrescentar
   coluna depende dela.

O material de apoio com as tabelas completas está em `materiais/aula09.html`, e as conclusões
desta página estão travadas em `tools/tests/test_problemas_aula09.py`.